In [38]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

# Load data
train = pd.read_csv('home-data-for-ml-course/train.csv')
test = pd.read_csv('home-data-for-ml-course/test.csv')

In [39]:
# Separate target
y = train.SalePrice
X = train.drop(['SalePrice'], axis=1)

In [ ]:
# Identify numerical and categorical columns
numeric_cols = [cname for cname in X.columns if X[cname].dtype in ['int64', 'float64']]
categorical_cols = [cname for cname in X.columns if X[cname].nunique() < 10 and X[cname].dtype == "object"]

In [41]:
# Preprocessing for numerical data
numerical_transformer = SimpleImputer(strategy='median')

In [42]:
numerical_transformer = SimpleImputer(strategy='median')

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
# Bundle preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

In [43]:
# Define the model
model = RandomForestRegressor(n_estimators=300, random_state=42)

# Create the final Pipeline
my_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                              ('model', model)
                             ])

In [44]:
# Cross-validate
scores = -1 * cross_val_score(my_pipeline, X, y, cv=5, scoring='neg_root_mean_squared_error')
print(f"Pipeline RMSE: {scores.mean():.4f}")

# Train and Predict
my_pipeline.fit(X, y)
preds = my_pipeline.predict(test)

Pipeline RMSE: 30559.8930


In [45]:
submission = pd.DataFrame({'Id': test['Id'], 'SalePrice': preds})
submission.to_csv('submission.csv', index=False)
submission.head()

,Id,SalePrice
0,1461,129955.500000
1,1462,156052.633333
2,1463,181759.373333
3,1464,181676.653333
4,1465,199802.383333
